# Analisis Keuangan — Fund Accounting

Dataset: `isi-database.json` — full Supabase export + 1 koreksi

- **Periode:** Jan–Jun 2026
- **Sumber:** Supabase project `iuran-asysyarif`
- **Tujuan:** Fund accounting — pisah Kas Besar vs Kas MDTA
- **Koreksi:** Feb 2026 ditambahkan alokasi Kas MDTA Rp 350.000 (lupa dicatat)


In [ ]:
import json
from collections import defaultdict
from datetime import datetime

# ── Load data ──
with open('isi-database.json', 'r', encoding='utf-8') as f:
    raw = json.load(f)

tables = raw['tables']
finances = tables['finances']['rows']
payments = tables['payments']['rows']
students = tables['students']['rows']

print(f'Loaded: {len(finances)} finances, {len(payments)} payments, {len(students)} students')
print()

# ── Klasifikasi transaksi (fund accounting) ──
# Returns (fund, category, is_transfer)
# fund:       'kas_besar' | 'kas_mdta'
# is_transfer: True = alokasi dana (bukan expense riil)

def classify(f):
    t = f['type']
    d = f.get('description', '').lower().strip()
    
    # ── Income ──
    if t == 'income':
        if 'uang suka rela' in d or 'potongan tabungan' in d or 'masuk ke kas mdta' in d:
            return ('kas_mdta', 'pemasukan_langsung_mdta', False)
        if 'infaq' in d:
            return ('kas_besar', 'infaq', False)
        return ('kas_besar', 'iuran_siswa', False)
    
    # ── Expense ──
    # Alokasi ke Kas MDTA → fund transfer (bukan expense riil)
    if 'kas mdta' in d:
        return ('kas_mdta', 'alokasi_mdta', True)
    
    # Gaji Guru → expense dari Kas Besar
    if d in ('guru', 'honor guru', 'pembayaran guru', 'honor guru', 'honor'):
        return ('kas_besar', 'gaji_guru', False)
    
    # Kas Mesjid → expense dari Kas Besar
    if 'kas mesjid' in d or 'kas masjid' in d:
        return ('kas_besar', 'kas_mesjid', False)
    
    # Sisanya (operasional: beli, fotokopi, print, seragam, dll) → expense dari Kas MDTA
    return ('kas_mdta', 'operasional', False)


# ── Terapkan klasifikasi ──
for f in finances:
    fund, cat, is_transfer = classify(f)
    f['_fund'] = fund
    f['_cat'] = cat
    f['_is_transfer'] = is_transfer


# ── Ringkasan ──
income_kb = sum(int(f['amount']) for f in finances if f['_fund'] == 'kas_besar' and f['type'] == 'income')
expense_kb = sum(int(f['amount']) for f in finances if f['_fund'] == 'kas_besar' and f['type'] == 'expense' and not f['_is_transfer'])
transfer_to_mdta = sum(int(f['amount']) for f in finances if f['_cat'] == 'alokasi_mdta')
income_mdta_direct = sum(int(f['amount']) for f in finances if f['_cat'] == 'pemasukan_langsung_mdta')
belanja_mdta = sum(int(f['amount']) for f in finances if f['_fund'] == 'kas_mdta' and f['type'] == 'expense' and not f['_is_transfer'])

saldo_kb = income_kb - expense_kb - transfer_to_mdta
saldo_mdta = transfer_to_mdta + income_mdta_direct - belanja_mdta

print(f'Pemasukan ke Kas Besar:         Rp {income_kb:>10,}')
print(f'Expense riil dari Kas Besar:    Rp {expense_kb:>10,}  (Gaji + Kas Mesjid)')
print(f'Fund transfer ke MDTA:          Rp {transfer_to_mdta:>10,}  (alokasi ke Kas MDTA)')
print(f'Pemasukan langsung MDTA:        Rp {income_mdta_direct:>10,}')
print(f'Belanja operasional dari MDTA:  Rp {belanja_mdta:>10,}  (spidol, fotokopi, dll)')
print(f'{"":->55}')
print(f'SALDO KAS BESAR:                Rp {saldo_kb:>10,}')
print(f'SALDO KAS MDTA:                 Rp {saldo_mdta:>10,}')
print(f'TOTAL UANG TUNAI:               Rp {saldo_kb + saldo_mdta:>10,}')


---
## Laporan per Bulan

Pemasukan, expense riil, fund transfer, dan saldo akhir per bulan.


In [ ]:
from collections import defaultdict
from datetime import datetime

monthly = defaultdict(lambda: {
    'income_kb': 0, 'expense_kb': 0, 'transfer_out': 0,
    'income_mdta': 0, 'belanja_mdta': 0
})

for f in finances:
    dt = f.get('date', f.get('created_at', ''))
    if not dt:
        continue
    d = datetime.fromisoformat(dt.replace('Z', ''))
    key = f'{d.year}-{d.month:02d}'
    m = monthly[key]
    amt = int(f['amount'])
    
    if f['type'] == 'income' and f['_fund'] == 'kas_besar':
        m['income_kb'] += amt
    elif f['_cat'] == 'pemasukan_langsung_mdta':
        m['income_mdta'] += amt
    elif f['_cat'] == 'alokasi_mdta':
        m['transfer_out'] += amt
    elif f['_fund'] == 'kas_besar' and f['type'] == 'expense' and not f['_is_transfer']:
        m['expense_kb'] += amt
    elif f['_fund'] == 'kas_mdta' and f['type'] == 'expense' and not f['_is_transfer']:
        m['belanja_mdta'] += amt

HEADER = f"{'Bulan':<10} {'Pemasukan':>12} {'Expense KB':>12} {'Trf MDTA':>10} {'Net KB':>12} {'Saldo KB':>12}"
print(HEADER)
print('-' * len(HEADER))

saldo_kb_run = 0
for month in sorted(monthly.keys()):
    m = monthly[month]
    net = m['income_kb'] - m['expense_kb'] - m['transfer_out']
    saldo_kb_run += net
    print(f"{month:<10} {m['income_kb']:>12,} {m['expense_kb']:>12,} {m['transfer_out']:>10,} {net:>12,} {saldo_kb_run:>12,}")

print()
print(f'Note: "Expense KB" hanya Gaji Guru + Kas Mesjid.')
print(f'      "Trf MDTA" = fund transfer ke Kas MDTA (bukan expense).')
print(f'      "Net KB" = Pemasukan - Expense - Transfer.')
print(f'      Belanja operasional (spidol, fotokopi) dibebankan ke Kas MDTA.')


---
## Tabel Kas MDTA — Akumulasi per Bulan

Alokasi dari Kas Besar, pemasukan langsung, belanja riil, dan sisa saldo.


In [ ]:
print(f"{'Bulan':<10} {'Alokasi KB':>12} {'Pemasukan':>12} {'Belanja':>12} {'Net MDTA':>12} {'Saldo MDTA':>12}")
print('-' * 70)

saldo_mdta_run = 0
for month in sorted(monthly.keys()):
    m = monthly[month]
    net = m['transfer_out'] + m['income_mdta'] - m['belanja_mdta']
    saldo_mdta_run += net
    print(f"{month:<10} {m['transfer_out']:>12,} {m['income_mdta']:>12,} {m['belanja_mdta']:>12,} {net:>12,} {saldo_mdta_run:>12,}")

print()
print(f'Note: "Alokasi KB" = fund transfer dari Kas Besar.')
print(f'      "Pemasukan" = langsung ke MDTA (uang sukarela).')
print(f'      "Belanja" = operasional riil dari Kas MDTA (spidol, fotokopi, print, seragam).')
print(f'      "Saldo MDTA" = saldo kumulatif dana MDTA.')


---
## Rincian Transaksi

Daftar semua transaksi dengan fund dan kategori.


In [ ]:
LABEL_FUND = {'kas_besar': '🏦 Kas Besar', 'kas_mdta': '📚 Kas MDTA'}
LABEL_CAT = {
    'iuran_siswa': '💳 Iuran Siswa', 'infaq': '💰 Infaq',
    'gaji_guru': '👨‍🏫 Gaji Guru', 'kas_mesjid': '🕌 Kas Mesjid',
    'alokasi_mdta': '📥 Alokasi MDTA',
    'operasional': '✏️ Operasional (MDTA)',
    'pemasukan_langsung_mdta': '📤 Pemasukan Langsung MDTA',
}

def get_date(f):
    dt = f.get('date', f.get('created_at', ''))
    return datetime.fromisoformat(dt.replace('Z', '')) if dt else datetime.min

sorted_finances = sorted(finances, key=get_date)

print(f"{'Tgl':<12} {'Fund':<22} {'Kategori':<30} {'Jumlah':>12}  {'Keterangan'}")
print('=' * 130)
for f in sorted_finances:
    d = get_date(f).strftime('%d/%m/%Y')
    fund = LABEL_FUND.get(f['_fund'], f['_fund'])
    cat = LABEL_CAT.get(f['_cat'], f['_cat'])
    amt = int(f['amount'])
    prefix = '+' if f['type'] == 'income' else '-'
    # Show transfer marker
    marker = ' 🔄' if f['_is_transfer'] else ''
    desc = (f.get('description', '') or '')[:68]
    print(f'{d:<12} {fund:<22} {cat:<30} {prefix}Rp {amt:>9,}{marker}  {desc}')


---
## Verifikasi

Cocokkan total antara pendekatan fund accounting vs pembukuan lama.


In [ ]:
# Pembukuan lama (semua income - semua expense)
old_total_income = sum(int(f['amount']) for f in finances if f['type'] == 'income')
old_total_expense = sum(int(f['amount']) for f in finances if f['type'] == 'expense')
old_balance = old_total_income - old_total_expense

# Fund accounting
income_kb = sum(int(f['amount']) for f in finances if f['_fund'] == 'kas_besar' and f['type'] == 'income')
expense_kb = sum(int(f['amount']) for f in finances if f['_fund'] == 'kas_besar' and f['type'] == 'expense' and not f['_is_transfer'])
transfer_mdta = sum(int(f['amount']) for f in finances if f['_cat'] == 'alokasi_mdta')
income_mdta_dir = sum(int(f['amount']) for f in finances if f['_cat'] == 'pemasukan_langsung_mdta')
belanja_mdta = sum(int(f['amount']) for f in finances if f['_fund'] == 'kas_mdta' and f['type'] == 'expense' and not f['_is_transfer'])

saldo_kb = income_kb - expense_kb - transfer_mdta
saldo_mdta = transfer_mdta + income_mdta_dir - belanja_mdta

print('PEMBUKUAN LAMA (semua income - semua expense):')
print(f'  Income:   Rp {old_total_income:,}')
print(f'  Expense:  Rp {old_total_expense:,}')
print(f'  Saldo:    Rp {old_balance:,}')
print()
print('FUND ACCOUNTING (pisah Kas Besar vs Kas MDTA):')
print(f'  Kas Besar: Rp {saldo_kb:,}  (income - gaji - mesjid - transfer)')
print(f'  Kas MDTA:  Rp {saldo_mdta:,}  (transfer + income_langsung - belanja)')
print(f'  Total:     Rp {saldo_kb + saldo_mdta:,}')
print()
print(f'Selisih: Rp {saldo_kb + saldo_mdta - old_balance:,}')
print(f'  -> fund transfer (Rp {transfer_mdta:,}) TIDAK dihitung sbg expense, ')
print(f'     tapi dipindah antar fund. Operasional (Rp {belanja_mdta:,}) tetap expense,')
print(f'     hanya dibebankan ke Kas MDTA bukan Kas Besar.')
